<a href="https://colab.research.google.com/github/birsensarem/Birsen_Data/blob/main/BirsenYC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Eğitim Analitiği Bağlamında Öğrenci Başarısının Veri Madenciliği ile İncelenmesi

Bu Notebook, UCI Student Performance veri seti kullanılarak öğrenci başarısının veri madenciliği yöntemleriyle incelenmesi amacıyla hazırlanmıştır. Çalışma veri ön işleme, keşifsel veri analizi, birliktelik kuralları, sınıflandırma, regresyon, kümeleme ve basit sinir ağı uygulamasını aynı eğitim analitiği problemi içinde birleştirir.

Araştırmanın Amacı: Öğrencilerin akademik başarı düzeylerinin devamsızlık, çalışma süresi, önceki başarısızlık durumu, aile desteği, okul desteği, internet erişimi ve sosyo-demografik değişkenler ile nasıl açıklanabileceğini ve tahmin edilebileceğini incelemektir.

In [ ]:
# Genel ayarlar
import warnings
warnings.filterwarnings("ignore")  # Gereksiz uyari mesajlarini gizler

RANDOM_STATE = 42  # Sonuclarin tekrar edilebilir olmasini saglar

## Veri Setinin Tanıtımı

UCI Student Performance veri seti, Portekiz'deki iki ortaöğretim okulundan elde edilen öğrenci başarı verilerini içerir. Veri setinde öğrencilerin demografik, sosyal, okul ve akademik değişkenleri yer alır. `G1`, `G2` ve `G3` değişkenleri dönem notlarını temsil eder. Bu Notebook'ta ana başarı göstergesi olarak `G3` kullanılacaktır.

Veri seti iki dosyadan oluşur: `student-mat.csv` Matematik dersi, `student-por.csv` Portekizce dersi verilerini içerir. Bu çalışmada varsayılan dosya olarak `student-mat.csv` kullanılmaktadır. İstenirse aynı kod akışı `student-por.csv` için de uygulanabilir.

Veri seti hem sınıflandırma hem de regresyon için uygundur. Ayrıca kümeleme ve birliktelik kuralları için de kullanılabilir. Çünkü sayısal ve kategorik değişkenler içermektedir.

In [ ]:
# Dataset source information
dataset_source = "https://archive.ics.uci.edu/dataset/320/student+performance"
dataset_zip_url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

print("Dataset source:", dataset_source)
print("Dataset zip URL:", dataset_zip_url)

Dataset source: https://archive.ics.uci.edu/dataset/320/student+performance
Dataset zip URL: https://archive.ics.uci.edu/static/public/320/student+performance.zip


## Kütüphanelerin Yüklenmesi

Bu bölümde veri analizi, görselleştirme, makine öğrenmesi, birliktelik kuralları ve sinir ağı için gerekli Python kütüphaneleri yüklenecektir.

In [ ]:
# If a package is missing, install it first:
# !pip install pandas numpy matplotlib seaborn scikit-learn mlxtend tensorflow ucimlrepo

import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# TensorFlow is used only in the neural network section.
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
except Exception as e:
    tf = None
    print("TensorFlow could not be imported. Neural network section may need package installation.")
    print(e)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Verinin İçe Aktarılması

Analize başlamadan önce veri setinin Python ortamına doğru biçimde aktarılması gerekir. Veri madenciliği sürecinde ilk teknik adım, veri setini doğru biçimde programa aktarmaktır. Veri yanlış okunursa eksik değer kontrolü, değişken türü incelemesi, grafikler ve modelleme aşamaları da hatalı ilerler.
df.head() çıktısında veri setinin ilk satırları ve değişken adları ayrı sütunlar hâlinde görünüyorsa veri doğru okunmuştur. Satırdaki bütün bilgiler tek bir sütunda toplanmış görünüyorsa dosya ayırıcı karakteri yanlış seçilmiş demektir.
Veri setinin içinde yer alan student-mat.csv dosyası kullanılacaktır. UCI Student Performance veri dosyalarında sütunlar noktalı virgülle ayrılmıştır. Bu nedenle veri okunurken sep=';' kullanılır. Ayırıcı doğru belirtilmezse Python tüm değişkenleri tek bir sütun gibi okuyabilir.





In [9]:

# Veri dosyasi Colab ortamına yuklendiği için doğrudan okunur
df = pd.read_csv("student-mat.csv", sep=";")

print("Veri boyutu:", df.shape)
df.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Veri boyutu: (395, 33)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag